# 実データからモデル入力まで：Paper 1 デモ

**目的：変数 `dataset` に読み込むところから、途中の表・ID列・テンソルを順に確認する。**

ローカルのYambda-50Mから、実在する2ユーザー（100・700）を説明用に抜き出す。学習・ダウンロード・新しい評価は行わず、既存のデータ・重み・結果も書き換えない。表示される件数はこのデモの件数で、研究全体の統計ではない。

上から **Run All** で実行できる。カーネルは **Paper 1 (.venv)** を選ぶ。必要な生データ・前処理済みファイルがない場合は、取得や再生成をせずエラーにする。

### 先に区別する2つのデータ

| 用途 | 1行の意味 | 入力 | 教師信号 |
|---|---|---|---|
| SASRecの表現学習 | ユーザーの再生列 | 過去の曲ID列 | 1曲ずらした「次の曲」＋ランダム負例 |
| 自発再聴の研究 | ユーザー×対象曲×推薦起点 | モデルごとの入力から得るスコア | 起点から24時間〜90日以内の自発再聴 `label_c` |

**自発再聴ラベルを、主比較のSASRecの次曲学習に入れるわけではない。** 別実験のEnd-to-End学習はこのデモの対象外。

流れ：`Parquet → LazyFrame → 2ユーザーの表 → 時刻・対象の確認 → 分岐（系列 / ペア / 疎行列 / 音響・記憶）→ モデルに渡せる入力`。

In [1]:
from pathlib import Path
import json
import sys
import platform

import numpy as np
import polars as pl
import torch
from IPython.display import display
from torch.utils.data import DataLoader

# リポジトリ直下・notebooks直下のどちらから起動してもよい。
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/relisten/config.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
from relisten import config
from relisten.labels import activity_counts, with_labels
from relisten.splits import assign_splits
from relisten.official_models import interaction_matrix
from relisten.sequence_model import PARAMETERS, _batch_inputs
from relisten.vendor.sasrec_data import TrainDataset, collate_fn
from relisten.vendor.sasrec_model import create_masked_tensor

np.random.seed(42)  # デモで生成する負例を再現する。
DEMO_USERS = [100, 700]  # 説明用の固定例。無作為標本ではない。
DAY = config.SECONDS_PER_DAY
MODEL_END_DAY = config.HEAD_REPRESENTATION_END_DAY
CUTOFF = (MODEL_END_DAY + 1) * DAY
MAX_SEQ_LEN = PARAMETERS["max_sequence_length"]
assert DAY == 86400 and MODEL_END_DAY == 29 and config.LOWER_BOUND_DAYS == 1
pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(12)
print({"root": str(ROOT), "python": platform.python_version(),
       "polars": pl.__version__, "torch": torch.__version__,
       "デモのテンソル": "CPU", "users": DEMO_USERS})

{'root': '/Users/macuser/dev/paper1', 'python': '3.13.13', 'polars': '1.44.2', 'torch': '2.14.1', 'デモのテンソル': 'CPU', 'users': [100, 700]}


## 1. データセットを変数に代入する

`pl.scan_parquet(...)` は **LazyFrame**。この時点では全再生をメモリに展開しない。後で必要な列・ユーザーを指定して `collect()` する。

実装対応：[`data.py / prepare_data`](../src/relisten/data.py)。ここでは同関数を呼ばず、読み込み部分だけを見せる。元の関数は中間ファイルを書き出すため。

In [2]:
RAW_LISTENS = config.RAW / "flat/50m/listens.parquet"
MODEL_DIR = config.PROCESSED / f"models/day_{MODEL_END_DAY}"
PAIR_PATH = config.PROCESSED / "pairs.parquet"
AUDIO_PATH = config.PROCESSED / "audio_embeddings_50m.parquet"
required_paths = [RAW_LISTENS, PAIR_PATH, AUDIO_PATH,
                  config.RAW / "manifest.json",
                  config.PROCESSED / "pair_provenance.json",
                  MODEL_DIR / "SASRec_items.npy",
                  MODEL_DIR / "SASRec_train_sequences.parquet"]
for path in required_paths:
    if not path.is_file():
        raise FileNotFoundError(f"既存ファイルが必要です（自動生成しません）: {path}")

manifest = json.loads((config.RAW / "manifest.json").read_text())
dataset = pl.scan_parquet(RAW_LISTENS)
print("dataset の型:", type(dataset).__name__)
print("固定データ revision:", manifest["revision"])
display(pl.DataFrame({"column": dataset.collect_schema().names(),
                      "dtype": [str(t) for t in dataset.collect_schema().dtypes()]}))

dataset の型: LazyFrame
固定データ revision: dd6f3a19eef5866e346c3270e098baa641a44948


column,dtype
str,str
"""uid""","""UInt32"""
"""timestamp""","""UInt32"""
"""item_id""","""UInt32"""
"""is_organic""","""UInt8"""
"""played_ratio_pct""","""UInt16"""
"""track_length_seconds""","""UInt32"""


| 元の列 | 今回の使い方 |
|---|---|
| `uid` | ユーザー単位の履歴・系列・評価 |
| `timestamp` | 秒。日番号、前後関係、経過時間を計算 |
| `item_id` | 曲を識別。SASRecでは連続した整数IDに変換 |
| `is_organic` | 0=推薦経由、1=自発経由。ラベルや音響プロフィールを作る |
| `played_ratio_pct` | SASRec・公式ベースラインの学習では50以上を採用 |
| `track_length_seconds` | 今回の研究実装では未使用 |

**timestampは5秒刻みの「秒値」で、5倍しない。** `day = timestamp // 86400`。古いAGENTS.mdの5倍という記述は、DECISIONS.md冒頭で訂正されている。

`likes.parquet` はRQ3の補助的な層別に使う。主モデル入力・自発再聴ラベルには使わないため、この経路では読み込まない。

In [3]:
raw_columns = ["uid", "timestamp", "item_id", "is_organic",
               "played_ratio_pct", "track_length_seconds"]
demo_listens = (
    dataset.filter(pl.col("uid").is_in(DEMO_USERS))
    .select(raw_columns)
    .with_columns(pl.col("timestamp").cast(pl.Int64))
    .sort("uid", "timestamp", "item_id")
    .collect()
)
print("読み込んだ表:", demo_listens.shape)
display(demo_listens.head(10))
display(demo_listens.group_by("uid").agg(
    pl.len().alias("events"), pl.col("item_id").n_unique().alias("unique_tracks"),
    pl.col("timestamp").min().alias("first_second"),
    pl.col("timestamp").max().alias("last_second"),
).sort("uid"))

読み込んだ表: (13719, 6)


uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
u32,i64,u32,u8,u16,u32
100,39420,1441281,0,100,105
100,39420,8326270,0,100,170
100,39625,286361,0,100,185
100,40110,732449,0,100,240
100,40360,3397170,0,46,130
100,40380,7849270,0,100,205
100,40640,1449307,0,100,205
100,40785,6887232,0,100,145
100,40885,2996197,0,100,95


uid,events,unique_tracks,first_second,last_second
u32,u32,u32,i64,i64
100,3585,892,39420,25966140
700,10134,2694,65835,25984705


## 2. スキーマ・値を確かめ、日番号を付ける

全経路・全再生率の履歴をまず保持する。最初に50%未満を削ると、自発再聴ラベルやACT-R用の履歴まで変わってしまう。

完全一致行や同時刻イベントの件数は表示するが、**このデモで勝手に重複削除・欠損補完はしない**。曲の重複再生は系列に残す。同時刻の異なる曲は本実装と同じ `timestamp, item_id` 順に置く。これは実際の再生順を復元したという意味ではない。

In [4]:
null_counts = demo_listens.null_count()
display(null_counts)
quality = {
    "完全一致する行の重複数（削除しない）": demo_listens.height - demo_listens.unique().height,
    "uid・timestamp重複数（別曲も含む）": demo_listens.height - demo_listens.unique(subset=["uid", "timestamp"]).height,
    "再生率50%未満": demo_listens.filter(pl.col("played_ratio_pct") < 50).height,
    "再生率100%超（ここでは除外しない）": demo_listens.filter(pl.col("played_ratio_pct") > 100).height,
}
print(quality)
assert null_counts.row(0) == (0,) * demo_listens.width
assert demo_listens["is_organic"].is_in([0, 1]).all()
assert (demo_listens["timestamp"] >= 0).all()

listens = demo_listens.with_columns(
    (pl.col("timestamp") // DAY).alias("day"),
    pl.when(pl.col("is_organic") == 1).then(pl.lit("自発")).otherwise(pl.lit("推薦")).alias("route"),
)
display(listens.select("uid", "timestamp", "day", "item_id", "route", "played_ratio_pct").head(10))

uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


{'完全一致する行の重複数（削除しない）': 5, 'uid・timestamp重複数（別曲も含む）': 1925, '再生率50%未満': 4398, '再生率100%超（ここでは除外しない）': 38}


uid,timestamp,day,item_id,route,played_ratio_pct
u32,i64,i64,u32,str,u16
100,39420,0,1441281,"""推薦""",100
100,39420,0,8326270,"""推薦""",100
100,39625,0,286361,"""推薦""",100
100,40110,0,732449,"""推薦""",100
100,40360,0,3397170,"""推薦""",46
100,40380,0,7849270,"""推薦""",100
100,40640,0,1449307,"""推薦""",100
100,40785,0,6887232,"""推薦""",100
100,40885,0,2996197,"""推薦""",100


## 3. モデル学習用の再生を切り出す

| 何を用意するか | モデルの学習期限 | 推薦起点の期間 |
|---|---|---|
| ヘッド学習用のスコア | 29日目まで | 30〜59日目 |
| 検証・テスト用のスコア | 149日目まで | 検証150〜164、テスト165〜210日目 |

このデモは**29日版**を追う。`timestamp < 30 * 86400` と `played_ratio_pct >= 50` を適用する。**推薦・自発の両方**を含む。

実装対応：[`sequence_model.py / train_sasrec`](../src/relisten/sequence_model.py)、[`light_models.py / history_before`](../src/relisten/light_models.py)。

In [5]:
model_events = (
    listens.filter((pl.col("timestamp") < CUTOFF) & (pl.col("played_ratio_pct") >= 50))
    .select("uid", "item_id", "timestamp")
    .sort("uid", "timestamp", "item_id")
)
assert model_events.height > 0 and model_events["timestamp"].max() < CUTOFF
display(pl.DataFrame({"stage": ["2ユーザーの全期間", "29日目まで・再生率制限なし", "29日目まで・50%以上"],
                      "rows": [listens.height, listens.filter(pl.col("timestamp") < CUTOFF).height, model_events.height]}))
display(model_events.head(10))

stage,rows
str,i64
"""2ユーザーの全期間""",13719
"""29日目まで・再生率制限なし""",897
"""29日目まで・50%以上""",767


uid,item_id,timestamp
u32,u32,i64
100,1441281,39420
100,8326270,39420
100,286361,39625
100,732449,40110
100,7849270,40380
100,1449307,40640
100,6887232,40785
100,2996197,40885
100,6474571,41130


## 4. 曲IDをモデル用の連続整数にする

元の曲IDは疎な識別子なので、そのままEmbeddingの行番号にしない。実験で保存済みの**29日版・全ユーザー共通の語彙**を読み、`1 ... N` に対応付ける。`0` はパディング用に予約する。

2ユーザーだけで新しく語彙を作り直さない。そうすると既存モデル・保存済み系列とIDの意味が変わる。ここで読む `.npy` は曲ID一覧で、モデルの重みではない。

In [6]:
items = np.load(MODEL_DIR / "SASRec_items.npy", allow_pickle=False)
assert items.ndim == 1 and np.all(items[1:] > items[:-1])
positions = np.searchsorted(items, model_events["item_id"].to_numpy())
assert (positions < len(items)).all()
assert np.array_equal(items[positions], model_events["item_id"].to_numpy())
indexed_events = model_events.with_columns(pl.Series("index", positions + 1))
print("29日版の全体語彙数:", len(items), "/ 0=padding")
display(indexed_events.head(10))
assert indexed_events["index"].min() >= 1
assert indexed_events["index"].max() <= len(items)

29日版の全体語彙数: 182700 / 0=padding


uid,item_id,timestamp,index
u32,u32,i64,i64
100,1441281,39420,28026
100,8326270,39420,161920
100,286361,39625,5524
100,732449,40110,14342
100,7849270,40380,152614
100,1449307,40640,28194
100,6887232,40785,133889
100,2996197,40885,58322
100,6474571,41130,126030


## 5. 1行＝1ユーザーの系列へまとめる

時刻順の曲IDをユーザーごとにまとめ、直近200イベントを残す。2イベント以上あるユーザーだけを採用する。ここで列名 `item_id` の中身は**元IDではなく、直前で作ったモデル用ID**。

保存済みの本実験の系列と、このデモで作り直した系列が一致することも確かめる。

In [7]:
sequences = (
    indexed_events.group_by("uid", maintain_order=True)
    .agg(pl.col("index").tail(MAX_SEQ_LEN).alias("item_id"))
    .filter(pl.col("item_id").list.len() >= 2)
)
saved_sequences = (
    pl.scan_parquet(MODEL_DIR / "SASRec_train_sequences.parquet")
    .filter(pl.col("uid").is_in(DEMO_USERS)).collect()
)
assert sequences.sort("uid").to_dicts() == saved_sequences.sort("uid").to_dicts()
display(sequences.select("uid", pl.col("item_id").list.len().alias("sequence_length"),
                         pl.col("item_id").list.head(8).alias("first_8"),
                         pl.col("item_id").list.tail(8).alias("last_8")))
print("既存の本実験の系列と完全一致")

uid,sequence_length,first_8,last_8
u32,u32,list[i64],list[i64]
100,200,"[50334, 120279, … 111035]","[54855, 128314, … 69837]"
700,200,"[138044, 111309, … 15448]","[169174, 133607, … 141282]"


既存の本実験の系列と完全一致


## 6. 公式TrainDataset → DataLoader → Tensor

例：系列 `[a, b, c, d]` は、入力 `[a, b, c]`、正例 `[b, c, d]` になる。負例は語彙 `1 ... N` から一様に生成する。**公式実装は、既に聴いた曲や正例との一致を負例から除外しない**。

実装対応：[`vendor/sasrec_data.py / TrainDataset, collate_fn`](../src/relisten/vendor/sasrec_data.py)。これらをそのまま呼ぶ。

本学習は `batch_size=256, shuffle=True, drop_last=True`。デモは2ユーザーなので、全員を見せるために `batch_size=2, shuffle=False, drop_last=False` とする。**学習条件の変更や新しい実験ではない。**

In [8]:
train_dataset = TrainDataset(sequences, num_items=len(items), max_seq_len=MAX_SEQ_LEN)
np.random.seed(42)
sample = train_dataset[0]
print("Datasetの長さ:", len(train_dataset))
display(pl.DataFrame({
    "入力item.ids": sample["item.ids"][:10],
    "次曲positive.ids": sample["positive.ids"][:10],
    "ランダムnegative.ids": sample["negative.ids"][:10],
}))
source_sequence = sequences.row(0, named=True)["item_id"]
assert sample["item.ids"] == source_sequence[:-1]
assert sample["positive.ids"] == source_sequence[1:]
assert sample["item.length"] == len(sample["positive.ids"])

Datasetの長さ: 2


入力item.ids,次曲positive.ids,ランダムnegative.ids
i64,i64,i64
50334,120279,121959
120279,96486,146868
96486,108635,131933
108635,139560,103695
139560,79380,119880
79380,163012,110269
163012,111035,54887
111035,168136,137338
168136,100224,168267


In [9]:
np.random.seed(42)
loader = DataLoader(train_dataset, batch_size=min(2, len(train_dataset)),
                    shuffle=False, drop_last=False, num_workers=0,
                    pin_memory=False, collate_fn=collate_fn)
batch = next(iter(loader))
display(pl.DataFrame([
    {"key": key, "shape": str(tuple(value.shape)), "dtype": str(value.dtype),
     "device": str(value.device), "first_values": str(value.flatten()[:8].tolist())}
    for key, value in batch.items()
]))
assert all(value.dtype == torch.int64 for value in batch.values())
assert int(batch["item.length"].sum()) == batch["item.ids"].numel()
assert batch["item.ids"].shape == batch["positive.ids"].shape == batch["negative.ids"].shape
print("ここまでで model(batch) に渡せる学習バッチが完成。forward/backwardは実行しない。")

key,shape,dtype,device,first_values
str,str,str,str,str
"""user.ids""","""(2,)""","""torch.int64""","""cpu""","""[100, 700]"""
"""user.length""","""(2,)""","""torch.int64""","""cpu""","""[1, 1]"""
"""item.ids""","""(398,)""","""torch.int64""","""cpu""","""[50334, 120279, 96486, 108635,…"
"""item.length""","""(2,)""","""torch.int64""","""cpu""","""[199, 199]"""
"""positive.ids""","""(398,)""","""torch.int64""","""cpu""","""[120279, 96486, 108635, 139560…"
"""positive.length""","""(2,)""","""torch.int64""","""cpu""","""[199, 199]"""
"""negative.ids""","""(398,)""","""torch.int64""","""cpu""","""[121959, 146868, 131933, 10369…"
"""negative.length""","""(2,)""","""torch.int64""","""cpu""","""[199, 199]"""


ここまでで model(batch) に渡せる学習バッチが完成。forward/backwardは実行しない。


### 平らなTensorから、paddingされた2次元配列へ

`collate_fn` の出力は長さを別に持つ1次元Tensor。モデル内部で `[batch_size, 最大系列長]` に整形し、`0` をpaddingに、`mask=True` を有効な要素にする。以下では公式の `create_masked_tensor` だけを呼び、モデルは作らない。

表示領域を小さくするため、同じ2系列の**末尾5曲と3曲**を使ってpaddingを可視化する。これは表示用の別変数で、完成した `batch` は変えない。

In [10]:
padded_ids, valid_mask = create_masked_tensor(batch["item.ids"], batch["item.length"])
assert padded_ids.shape == valid_mask.shape
assert torch.equal(padded_ids[valid_mask], batch["item.ids"])
assert torch.all(padded_ids[~valid_mask] == 0)
print("実バッチのpadding後:", tuple(padded_ids.shape), "/ 有効要素:", int(valid_mask.sum()))

visual_sequences = [train_dataset[0]["item.ids"][-5:], train_dataset[-1]["item.ids"][-3:]]
visual_inputs = _batch_inputs(visual_sequences, "cpu")
visual_padded, visual_mask = create_masked_tensor(visual_inputs["item.ids"], visual_inputs["item.length"])
print("表示用の短い例（0がpadding）:\n", visual_padded)
print("対応するmask:\n", visual_mask)
assert not visual_mask[1, -1] and visual_padded[1, -1] == 0

実バッチのpadding後: (2, 199) / 有効要素: 398
表示用の短い例（0がpadding）:
 tensor([[ 15249,   5975,  24046,  42493, 157492],
        [150217,  89470, 169222,      0,      0]])
対応するmask:
 tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True, False, False]])


## 7. 研究用のペア表：推薦起点を元データから選ぶ

ここからはSASRecの次曲学習とは別の経路。全再生率の `listens` を使い、研究で採点する起点を作る。

- **T2**：観測範囲でユーザーがその曲に初めて接触した時点が推薦経由。同じ初接触時刻に推薦・自発が混在したペアは除く。
- **T1**：分割ごとの最初の推薦時点より、厳密に前にその曲の再生がある。
- 推薦起点の日番号から train / validation / test を付ける。

実装対応：[`labels.py / build_pairs`](../src/relisten/labels.py)、[`splits.py / assign_splits`](../src/relisten/splits.py)。SQLの起点選択と同じ処理を、途中の表が見えるPolars操作で示す。

In [11]:
firsts = listens.group_by("uid", "item_id").agg(pl.col("timestamp").min().alias("first_timestamp"))
first_contacts = (
    listens.join(firsts, on=["uid", "item_id"])
    .filter(pl.col("timestamp") == pl.col("first_timestamp"))
    .group_by("uid", "item_id", "timestamp")
    .agg(pl.col("is_organic").min().alias("route_min"),
         pl.col("is_organic").max().alias("route_max"),
         pl.col("played_ratio_pct").min())
    .rename({"timestamp": "anchor_timestamp"})
    .with_columns((pl.col("anchor_timestamp") // DAY).alias("anchor_day"))
)
t2_anchors = assign_splits(first_contacts).filter(
    (pl.col("route_min") == 0) & (pl.col("route_max") == 0)
).with_columns(pl.lit("T2").alias("task"))

recommendations = assign_splits(
    listens.filter(pl.col("is_organic") == 0)
    .rename({"timestamp": "anchor_timestamp", "day": "anchor_day"})
)
t1_anchors = (
    recommendations.sort("uid", "item_id", "split", "anchor_timestamp", "played_ratio_pct")
    .group_by("uid", "item_id", "split", maintain_order=True)
    .agg(pl.col("anchor_timestamp").first(), pl.col("played_ratio_pct").first())
    .join(firsts, on=["uid", "item_id"])
    .filter(pl.col("first_timestamp") < pl.col("anchor_timestamp"))
    .with_columns((pl.col("anchor_timestamp") // DAY).alias("anchor_day"), pl.lit("T1").alias("task"))
)
anchor_columns = ["uid", "item_id", "anchor_timestamp", "played_ratio_pct", "split", "task", "anchor_day"]
anchors = pl.concat([t1_anchors.select(anchor_columns), t2_anchors.select(anchor_columns)])
display(anchors.sort("anchor_timestamp").head(10))
display(anchors.group_by("task", "split").len().sort("task", "split"))

uid,item_id,anchor_timestamp,played_ratio_pct,split,task,anchor_day
u32,u32,i64,u16,str,str,i64
700,6845741,2618925,100,"""train""","""T1""",30
700,9123015,2619095,100,"""train""","""T1""",30
700,35009,2619260,100,"""train""","""T2""",30
700,4463942,2619440,100,"""train""","""T2""",30
700,8111951,2619665,100,"""train""","""T2""",30
700,7981655,2619900,100,"""train""","""T1""",30
700,4277760,2620145,100,"""train""","""T2""",30
700,5690419,2620835,100,"""train""","""T2""",30
700,4856971,2621040,100,"""train""","""T2""",30


task,split,len
str,str,u32
"""T1""","""test""",524
"""T1""","""train""",312
"""T1""","""validation""",382
"""T2""","""test""",226
"""T2""","""train""",202
"""T2""","""validation""",192


### 保存済みの起点と照合し、追跡情報を読む

`pairs.parquet` には、起点から90日までの再聴時刻の一覧、追跡の充足、活動日数が保存されている。ここは**読み込みだけ**。直後に1ペアの再聴時刻と活動日数を生データから作り直して照合する。

起点選択の2ユーザー全件も、保存済みの本実験データに一致するか確認する。

In [12]:
saved_pairs = (
    pl.scan_parquet(PAIR_PATH)
    .filter(pl.col("uid").is_in(DEMO_USERS) & pl.col("task").is_in(["T1", "T2"]))
    .collect()
)
keys = ["task", "split", "uid", "item_id", "anchor_timestamp"]
assert anchors.select(anchor_columns).sort(keys).to_dicts() == saved_pairs.select(anchor_columns).sort(keys).to_dicts()

labeled_pairs = with_labels(saved_pairs, lower_days=1, horizon_days=90, min_active_days=30)
display(labeled_pairs.select(*keys, "active_days_l1_h90", "label_b", "label_c").head(10))
print("起点の再構成が保存済み全件と一致:", anchors.height, "ペア")
print("完全追跡＋活動30日以上を満たすペア:", labeled_pairs.height)

task,split,uid,item_id,anchor_timestamp,active_days_l1_h90,label_b,label_c
str,str,u32,u32,i64,i16,u8,u8
"""T1""","""test""",100,27230,18025175,52,1,0
"""T1""","""test""",100,155792,15096700,34,1,1
"""T1""","""test""",100,289597,15093065,34,1,1
"""T1""","""test""",100,308692,17749780,52,1,0
"""T1""","""test""",100,593420,17763775,52,1,0
"""T1""","""test""",100,645242,15094765,34,1,0
"""T1""","""test""",100,707462,15098325,34,0,0
"""T1""","""test""",100,911737,15147905,34,0,0
"""T1""","""test""",100,959813,15149570,34,1,0


起点の再構成が保存済み全件と一致: 1838 ペア
完全追跡＋活動30日以上を満たすペア: 1816


## 8. 1ペアの自発再聴ラベルを生データで追う

`L=1` の正例窓は **[起点+24時間, 起点+90日]（両端を含む）**。

Bは経路を問わない再聴、Cは自発経由の再聴。最初の再聴が24時間未満でも、その後に窓内の再聴があれば正例なので、最初の再聴時刻だけで判定しない。

以下は説明しやすいように、**train / T2 / C正例 / 29日版の既知曲**を1つ選ぶ。例の選択にラベルを使っているだけで、モデル入力やモデル選択には使わない。

In [13]:
known_items = pl.DataFrame({"item_id": items})
example_candidates = (
    labeled_pairs.filter((pl.col("split") == "train") & (pl.col("task") == "T2") & (pl.col("label_c") == 1))
    .join(known_items, on="item_id", how="semi").sort("uid", "anchor_timestamp", "item_id")
)
assert example_candidates.height > 0
example = example_candidates.row(0, named=True)
uid, target_item, t0 = example["uid"], example["item_id"], example["anchor_timestamp"]
returns = (
    listens.filter((pl.col("uid") == uid) & (pl.col("item_id") == target_item)
                   & (pl.col("timestamp") > t0) & (pl.col("timestamp") <= t0 + 90 * DAY))
    .with_columns((pl.col("timestamp") - t0).alias("elapsed_seconds"))
    .with_columns((pl.col("elapsed_seconds") / DAY).alias("elapsed_days"),
                  pl.col("elapsed_seconds").is_between(DAY, 90 * DAY).alias("in_label_window"))
)
print({"uid": uid, "item_id": target_item, "anchor_second": t0, "anchor_day": t0 // DAY,
       "label_window_seconds": [t0 + DAY, t0 + 90 * DAY]})
display(returns.select("timestamp", "route", "played_ratio_pct", "elapsed_seconds", "elapsed_days", "in_label_window"))
actual_b = sorted(returns["elapsed_seconds"].unique().to_list())
actual_c = sorted(returns.filter(pl.col("is_organic") == 1)["elapsed_seconds"].unique().to_list())
assert actual_b == example["relisten_seconds_b"]
assert actual_c == example["relisten_seconds_c"]
manual_b = int(any(DAY <= s <= 90 * DAY for s in actual_b))
manual_c = int(any(DAY <= s <= 90 * DAY for s in actual_c))
assert (manual_b, manual_c) == (example["label_b"], example["label_c"])
print("生データからの判定:", {"label_b": manual_b, "label_c": manual_c})

{'uid': 100, 'item_id': 5083803, 'anchor_second': 3243120, 'anchor_day': 37, 'label_window_seconds': [3329520, 11019120]}


timestamp,route,played_ratio_pct,elapsed_seconds,elapsed_days,in_label_window
i64,str,u16,i64,f64,bool
4116830,"""推薦""",100,873710,10.112384,true
5095105,"""自発""",100,1851985,21.435012,true


生データからの判定: {'label_b': 1, 'label_c': 1}


### 追跡不足と活動条件も確認する

活動日数は、同じ経過時間窓内で**全経路の再生が1回以上ある暦日**の数。再生率は制限しない。将来の活動情報は対象の選択に使うが、予測入力には渡さない。

追跡可能かどうかはユーザーの最終再生時刻ではなく、全データの観測終了時刻と比較する。

In [14]:
window_activity = listens.filter(
    (pl.col("uid") == uid) & pl.col("timestamp").is_between(t0 + DAY, t0 + 90 * DAY)
)
manual_active_days = window_activity["day"].n_unique()
user_days = listens.group_by("uid", "day").agg(
    pl.col("timestamp").min().alias("first_timestamp"),
    pl.col("timestamp").max().alias("last_timestamp"),
)
example_frame = example_candidates.head(1)
count_from_function = int(activity_counts(example_frame, user_days, 1, 90)[0])
assert manual_active_days == count_from_function == example["active_days_l1_h90"]
max_timestamp = json.loads((config.PROCESSED / "pair_provenance.json").read_text())["max_timestamp"]
assert (t0 + 90 * DAY <= max_timestamp) == example["complete_followup_90"]
print({"生データから数えた活動暦日": manual_active_days,
       "30日以上": manual_active_days >= 30,
       "90日完全追跡": example["complete_followup_90"],
       "全データの観測終了秒": max_timestamp})

{'生データから数えた活動暦日': 51, '30日以上': True, '90日完全追跡': True, '全データの観測終了秒': 26000000}


## 9. 同じ推薦起点を、SASRecの採点入力へ変換する

表現の学習データは29日目で固定。一方、推薦時点の系列は**起点より前の履歴**まで更新する。この2つの期限を混同しない。

本実装は、50%以上の再生から29日版で既知の曲だけを残し、同時刻イベントをすべて除く。対象の推薦再生自身が50%以上なら、対象曲を最後に**1回だけ**追加し、直近200曲にする。未来の再聴は含めない。

実装対応：[`sequence_model.py / build_sasrec_features, _batch_inputs`](../src/relisten/sequence_model.py)。

In [15]:
past_for_sasrec = listens.filter(
    (pl.col("uid") == uid) & (pl.col("timestamp") < t0) & (pl.col("played_ratio_pct") >= 50)
).sort("timestamp", "item_id")
known_past = past_for_sasrec.join(known_items, on="item_id", how="semi").sort("timestamp", "item_id")
past_indices = (np.searchsorted(items, known_past["item_id"].to_numpy()) + 1).tolist()
target_index = int(np.searchsorted(items, target_item)) + 1
inference_sequence = past_indices[-MAX_SEQ_LEN:]
include_anchor = example["played_ratio_pct"] >= 50
if include_anchor:
    inference_sequence = (inference_sequence + [target_index])[-MAX_SEQ_LEN:]
assert inference_sequence
assert known_past["timestamp"].max() < t0
if include_anchor:
    assert inference_sequence[-1] == target_index
inference_inputs = _batch_inputs([inference_sequence], "cpu")
target_indices = torch.tensor([target_index], dtype=torch.long)
display(known_past.select("uid", "timestamp", "item_id", "played_ratio_pct").tail(8))
print({"既知の過去イベント数": known_past.height,
       "起点の推薦自身を追加": include_anchor,
       "完成系列の長さ": len(inference_sequence), "末尾8曲のモデルID": inference_sequence[-8:],
       "対象のモデルID": target_index})
print("inference_inputs:", {k: (tuple(v.shape), str(v.dtype)) for k, v in inference_inputs.items()})
print("target_indices:", target_indices)
print("未知の対象曲は0をEmbeddingに入れて採点せず、本実装では別途最低スコアにする。")

uid,timestamp,item_id,played_ratio_pct
u32,i64,u32,u16
100,3241560,1643756,100
100,3241655,1990771,100
100,3242090,2219570,100
100,3242095,8490756,100
100,3242510,5699545,100
100,3242510,8955973,100
100,3242715,7610592,100
100,3243000,2277400,100


{'既知の過去イベント数': 307, '起点の推薦自身を追加': True, '完成系列の長さ': 200, '末尾8曲のモデルID': [38801, 43253, 165093, 111035, 174226, 147943, 44343, 98999], '対象のモデルID': 98999}
inference_inputs: {'item.ids': ((200,), 'torch.int64'), 'item.length': ((1,), 'torch.int64')}
target_indices: tensor([98999])
未知の対象曲は0をEmbeddingに入れて採点せず、本実装では別途最低スコアにする。


## 10. 他モデルでは、完成するデータの形が違う

### iALS / BPR：ユーザー×曲の二値疎行列

同じ29日目まで・50%以上の `model_events` から作る。同一ユーザー・同一曲の複数再生を1にまとめ、再生順は保持しない。SASRecの系列では繰り返しを残していたことと対比する。

本実装の [`official_models.py / interaction_matrix`](../src/relisten/official_models.py) を呼ぶ。ここでは2ユーザー内の曲に限った説明用の行列で、全体モデルの学習には使わない。

In [16]:
matrix_users, matrix_items, interaction_csr = interaction_matrix(model_events, binary=True)
assert interaction_csr.shape == (len(matrix_users), len(matrix_items))
assert np.all(interaction_csr.data == 1)
assert interaction_csr.nnz == model_events.unique(subset=["uid", "item_id"]).height
print({"type": type(interaction_csr).__name__, "shape": interaction_csr.shape,
       "dtype": str(interaction_csr.dtype), "nonzero": interaction_csr.nnz})
preview = pl.DataFrame(interaction_csr[:, :8].toarray(), schema=[f"item_{int(i)}" for i in matrix_items[:8]])
display(preview.with_columns(pl.Series("uid", matrix_users)).select("uid", pl.exclude("uid")))

{'type': 'csr_matrix', 'shape': (2, 581), 'dtype': 'float32', 'nonzero': 581}


uid,item_86612,item_133483,item_134877,item_155475,item_173016,item_197453,item_246465,item_253722
u32,f32,f32,f32,f32,f32,f32,f32,f32
100,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0
700,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0


### AudioCosine / ACT-R：起点より前の履歴＋音響ベクトル

SASRecのID Embedding（学習する64次元）と、Yambda提供の音響Embedding（128次元）は別物。

- 音響ベクトルは既存キャッシュの `normalized_embed` を読む。ここでは必要な曲だけを取得する。
- AudioCosineのプロフィールは、起点より厳密に前の**自発経由の各イベント**のベクトル平均を正規化したもの。同じ曲を複数回聴けば複数回含む。再生率の制限はない。
- ACT-Rは同じ全経路の過去履歴、直前1曲、音響ベクトル、学習期限までのセッション共起を使う。PyTorchの次曲学習バッチに変換するモデルではない。

実装対応：[`light_models.py / _user_audio_actr, fit_item_sessions`](../src/relisten/light_models.py)。ここでは入力を確認し、全体の共起行列やACT-Rスコアを再計算しない。

In [17]:
past_history = listens.filter((pl.col("uid") == uid) & (pl.col("timestamp") < t0)).sort("timestamp", "item_id")
organic_history = past_history.filter(pl.col("is_organic") == 1)
needed_items = sorted(set(past_history["item_id"].to_list() + [target_item]))
audio_dataset = pl.scan_parquet(AUDIO_PATH)
audio_table = audio_dataset.filter(pl.col("item_id").is_in(needed_items)).collect().sort("item_id")
audio_vectors = np.stack(audio_table["normalized_embed"].to_numpy()).astype(np.float32)
assert audio_vectors.shape == (audio_table.height, 128)
assert np.isfinite(audio_vectors).all()
assert np.allclose(np.linalg.norm(audio_vectors, axis=1), 1.0, atol=1e-4)
print("音響ベクトル:", audio_vectors.shape, audio_vectors.dtype)
display(audio_table.select("item_id", pl.col("normalized_embed").list.head(6).alias("first_6_of_128")))

embedding_by_item = dict(zip(audio_table["item_id"].to_list(), audio_vectors))
organic_vectors = np.stack([embedding_by_item.get(item, np.zeros(128, dtype=np.float32))
                            for item in organic_history["item_id"].to_list()])
profile_sum = organic_vectors.sum(axis=0)
profile_norm = np.linalg.norm(profile_sum)
assert profile_norm > 0  # このデモの例には音響付きの自発履歴がある。
audio_profile = profile_sum / profile_norm
print("AudioCosine用プロフィール:", audio_profile.shape, audio_profile.dtype)
print("対象曲ベクトル:", embedding_by_item[target_item].shape)

context_item = past_history["item_id"][-1]
actr_input_history = past_history.select("uid", "timestamp", "item_id", "is_organic").with_columns(
    ((t0 + config.LOWER_BOUND_DAYS * DAY - pl.col("timestamp")) / DAY).alias("age_days_at_label_start")
)
assert (actr_input_history["age_days_at_label_start"] >= 1).all()
print({"ACT-R直前曲": int(context_item), "履歴行数": actr_input_history.height,
       "活性の参照秒": t0 + DAY,
       "T2の成分": "S+P（Bを使わない）",
       "共起の学習期限": MODEL_END_DAY})
display(actr_input_history.tail(8))

音響ベクトル: (219, 128) float32


item_id,first_6_of_128
u32,list[f32]
86612,"[-0.150445, -0.019578, … -0.192915]"
134877,"[-0.176959, -0.10412, … -0.106146]"
197453,"[-0.113141, -0.028659, … -0.144452]"
246465,"[-0.118891, -0.081801, … -0.169179]"
253722,"[-0.162103, -0.079874, … 0.045491]"
286361,"[-0.083622, 0.014928, … -0.07105]"
…,…
9080942,"[-0.070017, -0.073553, … -0.184857]"
9082118,"[-0.126361, -0.054714, … -0.08457]"


AudioCosine用プロフィール: (128,) float32
対象曲ベクトル: (128,)
{'ACT-R直前曲': 2277400, '履歴行数': 370, '活性の参照秒': 3329520, 'T2の成分': 'S+P（Bを使わない）', '共起の学習期限': 29}


uid,timestamp,item_id,is_organic,age_days_at_label_start
u32,i64,u32,u8,f64
100,3242095,8490756,1,1.011863
100,3242510,4280471,1,1.00706
100,3242510,5699545,1,1.00706
100,3242510,8955973,1,1.00706
100,3242530,7371186,1,1.006829
100,3242715,7610592,0,1.0046875
100,3242730,7621980,0,1.004514
100,3243000,2277400,0,1.001389


## 11. 共通ヘッドに渡す直前のXとyも見る

最後は、保存済みのモデルスコアとラベルをペアのキーで結合する。**ヘッドはここで学習しない。**

`score_only` なら `X` はスコア1列。`baseline_plus_score` なら、ユーザーの過去再聴率・過去活動日数の2列を加える。標準化はヘッド学習時にtrainだけでfitするため、このセルではまだ実施しない。

主指標はヘッドを通さない生スコアのユーザー内AUC。ヘッド経由の値は副指標。ここでモデル間のAUCや精度は算出しない。

実装対応：[`evaluation.py / common_head`](../src/relisten/evaluation.py)、[`score_protocol.py`](../src/relisten/score_protocol.py)。

In [18]:
feature_table = (
    pl.scan_parquet(config.PROCESSED / "light_features.parquet")
    .filter(pl.col("uid").is_in(DEMO_USERS))
    .select(*keys, "model_end_day", "audio_cosine", "past_return_rate", "past_active_days")
    .collect()
)
head_dataset = labeled_pairs.filter((pl.col("task") == "T2") & (pl.col("split") == "train")).join(
    feature_table, on=keys, how="left", validate="1:1"
).sort("uid", "anchor_timestamp", "item_id")
X_score_only = head_dataset.select("audio_cosine").to_numpy()
X_baseline_plus_score = head_dataset.select("past_return_rate", "past_active_days", "audio_cosine").to_numpy()
y = head_dataset["label_c"].to_numpy()
groups = head_dataset["uid"].to_numpy()
assert np.isfinite(X_baseline_plus_score).all()
assert (head_dataset["model_end_day"] == MODEL_END_DAY).all()
assert len(X_score_only) == len(y) == len(groups)
display(head_dataset.select(*keys, "audio_cosine", "past_return_rate", "past_active_days", "label_c").head(8))
print({"X_score_only": X_score_only.shape, "X_baseline_plus_score": X_baseline_plus_score.shape,
       "y": y.shape, "y_dtype": str(y.dtype), "groups": groups.shape})
print("Xとyは別変数。未来の再聴時刻・活動日数・label_cをXに混ぜない。")

task,split,uid,item_id,anchor_timestamp,audio_cosine,past_return_rate,past_active_days,label_c
str,str,u32,u32,i64,f64,f64,i64,u8
"""T2""","""train""",100,9039786,2815110,0.72465,0.043319,12,0
"""T2""","""train""",100,5919606,2815250,0.7358,0.043319,12,0
"""T2""","""train""",100,5450885,2815530,0.653518,0.043319,12,0
"""T2""","""train""",100,1012904,2815735,0.763897,0.043319,12,0
"""T2""","""train""",100,2972508,2998695,0.809229,0.043319,12,0
"""T2""","""train""",100,7610592,3242715,0.850734,0.043319,12,0
"""T2""","""train""",100,7621980,3242730,0.66653,0.043319,12,0
"""T2""","""train""",100,2277400,3243000,0.737923,0.043319,12,0


{'X_score_only': (202, 1), 'X_baseline_plus_score': (202, 3), 'y': (202,), 'y_dtype': 'uint8', 'groups': (202,)}
Xとyは別変数。未来の再聴時刻・活動日数・label_cをXに混ぜない。


## 最後に：完成した変数と対応する本実装

| 変数 | 完成したもの | 次に渡す先 |
|---|---|---|
| `dataset` | 生Parquetを参照するLazyFrame | 必要列・対象ユーザーの選択 |
| `model_events` | 29日目まで・50%以上の再生表 | 系列・疎行列・人気度の生成 |
| `sequences` / `train_dataset` | ユーザーごとの直近200曲・次曲教師信号 | 公式SASRec DataLoader |
| `batch` | 学習用Tensor辞書（int64） | `SASRecEncoder.forward` の学習経路 |
| `labeled_pairs` | T1/T2の起点・追跡条件・B/Cラベル | 研究用の評価・ヘッド学習 |
| `inference_inputs` / `target_indices` | 推薦起点での系列と対象ID | SASRecの採点経路 |
| `interaction_csr` | ユーザー×曲の二値疎行列 | iALS / BPR |
| `audio_table` / `audio_profile` | 128次元音響ベクトル・過去自発プロフィール | AudioCosine、ACT-Rの部分一致 |
| `actr_input_history` | 厳密に起点前の全経路履歴と参照時点でのage | ACT-R。共起情報は別途期限内で固定 |
| `X_score_only` / `X_baseline_plus_score` / `y` | 保存済みスコアから作るヘッド入力と教師ラベル | `common_head`（このデモでは呼ばない） |

### このデモで確認したこと

- 生データから作り直したSASRec系列が、本実験の保存済み系列と一致する。
- 生データから作り直したT1/T2の起点が、本実験の保存済み起点と一致する。
- 1ペアの再聴時刻一覧・B/Cラベル・活動日数・追跡充足が、保存済み値と一致する。
- Tensorのshape・dtype・padding、疎行列の二値性、音響ベクトルの128次元・正規化を検証する。

### 範囲と限界

全件の前処理の再実行、モデルの初期化・forward・学習、精度評価はしていない。全データの品質をこの2ユーザーで保証するものではない。学習期限29日版のID語彙と、保存済みペア・音響・スコアを利用しているため、空の環境で生データだけから全成果物を再生成するNotebookではない。

既存の実験手順：`00_data.ipynb → 01_labels_splits.ipynb → 03_light_baselines.ipynb / 04_*.ipynb`。研究全体の数値は [`docs/results_factsheet.md`](../docs/results_factsheet.md)、仕様の訂正・判断は [`DECISIONS.md`](../DECISIONS.md) を参照。

In [19]:
print("デモ完了：保存済みデータとの照合・入力のshape/dtypeチェックを通過。学習・評価・既存ファイルの保存は未実行。")

デモ完了：保存済みデータとの照合・入力のshape/dtypeチェックを通過。学習・評価・既存ファイルの保存は未実行。
